# Day-ahead purchase volume under imbalance prices

We supply about 1% of national load. Each hour we buy a volume `q` in the day-ahead auction. If
our customers use more than `q` we are short and buy the difference at the system buy price
(above day-ahead); if they use less we sell the surplus at the system sell price (below
day-ahead). The question is how much to buy relative to the load forecast.

Data: `hourly_power_clean.csv`. Imbalance prices are not in the file, so they are simulated from
the day-ahead price with the typical premium / discount observed on the market.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge, QuantileRegressor

pd.set_option("display.width", 120)

## Load data and imbalance prices

In [2]:
df = pd.read_csv("../data/hourly_power_clean.csv", parse_dates=["time"]).set_index("time")
SHARE = 0.01

rng = np.random.default_rng(42)
n = len(df)
premium = rng.lognormal(np.log(25) - 0.5 * 0.6**2, 0.6, n)     # mean 25 EUR/MWh
discount = rng.lognormal(np.log(15) - 0.5 * 0.6**2, 0.6, n)    # mean 15 EUR/MWh
imb = pd.DataFrame({
    "da_price": df["price_eur_mwh"],
    "sys_buy": df["price_eur_mwh"] + premium,
    "sys_sell": df["price_eur_mwh"] - discount,
}, index=df.index)
imb.index = imb.index.tz_convert("Europe/London").tz_localize(None)   # local time for reporting
imb[["da_price", "sys_buy", "sys_sell"]].describe().round(1)

,da_price,sys_buy,sys_sell
count,17520.0,17520.0,17520.0
mean,98.5,123.5,83.3
std,36.9,40.4,38.2
min,-19.9,-7.9,-77.6
25%,73.5,96.1,57.8
50%,97.7,122.1,82.8
75%,122.8,149.1,108.5
max,419.6,446.8,408.8


## Load forecast

In [3]:
cons = df["consumption_mwh"]
X = pd.DataFrame({
    "lag24": cons.shift(24),
    "lag168": cons.shift(168),
    "temp": df["temp_c"],
    "hour": df.index.hour,
    "dow": df.index.dayofweek,
})
X = pd.get_dummies(X, columns=["hour", "dow"], dtype=float)
data = X.assign(y=cons).dropna()
features = [c for c in data.columns if c != "y"]

model = Ridge(alpha=1.0).fit(data[features], data["y"])
data["fc"] = model.predict(data[features])
data["resid"] = data["y"] - data["fc"]

train = data.loc["2022"]
test = data.loc["2023"]
print(f"train resid sd {train['resid'].std():.0f} MWh, test resid sd {test['resid'].std():.0f} MWh")

train resid sd 899 MWh, test resid sd 888 MWh


## Optimal purchase quantile

Classic newsvendor: with under-purchase cost `c_u` (premium) and over-purchase cost `c_o`
(discount), the optimal quantity is the `c_o / (c_u + c_o)` quantile of the load distribution.

In [4]:
c_u = premium.mean()
c_o = discount.mean()
q_star = c_o / (c_u + c_o)
adj = np.quantile(train["resid"], q_star)
print(f"c_u = {c_u:.1f}, c_o = {c_o:.1f}, q* = {q_star:.3f}")
print(f"safety margin at q*: {adj:.1f} MWh")

c_u = 25.0, c_o = 15.2, q* = 0.378
safety margin at q*: -222.1 MWh


Build the 2023 book. Forecast and load are converted to our share; the margin is scaled with them.

In [5]:
book = pd.DataFrame({
    "load": test["y"] * SHARE,
    "fc": test["fc"] * SHARE,
})
book["q"] = book["fc"] + adj / 100
book.index = book.index.tz_localize(None)
book = book.join(imb, how="inner")
print(len(book), "hours")
book.head()

8760 hours


,load,fc,q,da_price,sys_buy,sys_sell
time,,,,,,
2023-01-01 00:00:00,269.987,268.399166,266.178171,70.53,119.427103,57.050689
2023-01-01 01:00:00,248.560,255.734778,253.513783,81.49,102.950424,68.947881
2023-01-01 02:00:00,248.656,250.760118,248.539122,72.15,112.339223,58.787021
2023-01-01 03:00:00,247.890,245.958063,243.737068,78.89,92.732264,55.647311
2023-01-01 04:00:00,251.856,247.787018,245.566023,78.55,127.113895,71.551479


## Imbalance cost

In [6]:
def imbalance_cost(q, load, sys_buy, sys_sell, da):
    short = np.maximum(q - load, 0)
    long_ = np.maximum(load - q, 0)
    return short * (sys_buy - da) + long_ * (da - sys_sell)

book["cost_q"] = imbalance_cost(book["q"], book["load"], book["sys_buy"], book["sys_sell"], book["da_price"])
book["cost_fc"] = imbalance_cost(book["fc"], book["load"], book["sys_buy"], book["sys_sell"], book["da_price"])
book[["cost_q", "cost_fc"]].sum().round(0)

cost_q     1167273.0
cost_fc    1249437.0
dtype: float64

Cost on the hours where we were actually short, by month:

In [7]:
short_hours = book[book["load"] > book["q"]]
short_hours["cost_q"].resample("MS").sum().round(0)

time
2023-01-01    66494.0
2023-02-01    55077.0
2023-03-01    48539.0
2023-04-01    35799.0
2023-05-01    32703.0
2023-06-01    48493.0
2023-07-01    79124.0
2023-08-01    62273.0
2023-09-01    38333.0
2023-10-01    40621.0
2023-11-01    37467.0
2023-12-01    38184.0
Freq: MS, Name: cost_q, dtype: float64

## Is a simple percentage margin better?

In [8]:
grid = np.arange(-0.05, 0.051, 0.01)
res = []
for m in grid:
    q = book["fc"] * (1 + m)
    res.append({"margin": round(m, 2),
                "cost": imbalance_cost(q, book["load"], book["sys_buy"], book["sys_sell"], book["da_price"]).sum()})
res = pd.DataFrame(res).set_index("margin")
res["vs_fc_%"] = (res["cost"] / res.loc[0.0, "cost"] - 1) * 100
res.round(0)

,cost,vs_fc_%
margin,,
-0.05,1944294.0,56.0
-0.04,1629387.0,30.0
-0.03,1371533.0,10.0
-0.02,1207717.0,-3.0
-0.01,1159387.0,-7.0
0.00,1249437.0,0.0
0.01,1477680.0,18.0
0.02,1833169.0,47.0
0.03,2286757.0,83.0


In [9]:
best_margin = res["cost"].idxmin()
saving = -res.loc[best_margin, "vs_fc_%"]
print(f"best margin {best_margin:+.0%}, saving {saving:.1f}% vs buying the forecast")

best margin -1%, saving 7.2% vs buying the forecast


## Quantile regression as an alternative

In [10]:
qr = QuantileRegressor(quantile=0.9, alpha=0.0, solver="highs").fit(train[features], train["y"])
test_qr = qr.predict(test[features])
rmse_mean = np.sqrt(np.mean((test["y"] - test["fc"]) ** 2))
rmse_qr = np.sqrt(np.mean((test["y"] - test_qr) ** 2))
print(f"RMSE mean model {rmse_mean:.0f}, RMSE quantile regression {rmse_qr:.0f}")

RMSE mean model 890, RMSE quantile regression 1564


## Monte Carlo on the imbalance cost

In [11]:
sims = []
for i in range(200):
    np.random.seed(42)
    r = np.random.choice(train["resid"].values, size=len(book), replace=True) * SHARE
    load_sim = book["fc"] + r
    q = book["fc"] * (1 + best_margin)
    sims.append(imbalance_cost(q, load_sim, book["sys_buy"], book["sys_sell"], book["da_price"]).sum())
sims = np.array(sims)
print(f"expected cost {sims.mean():,.0f}, 95% CI [{np.percentile(sims, 2.5):,.0f}, {np.percentile(sims, 97.5):,.0f}]")

expected cost 1,196,217, 95% CI [1,196,217, 1,196,217]


Cost per euro of day-ahead price, to check the cost is not just tracking the price level:

In [12]:
book["cost_per_eur"] = book["cost_q"] / book["da_price"]
book["cost_per_eur"].resample("MS").mean().round(3)

time
2023-01-01    1.032
2023-02-01    1.274
2023-03-01   -1.469
2023-04-01    1.823
2023-05-01    2.399
2023-06-01    2.991
2023-07-01    2.674
2023-08-01    2.297
2023-09-01    2.577
2023-10-01    1.744
2023-11-01    1.574
2023-12-01    1.595
Freq: MS, Name: cost_per_eur, dtype: float64

## Results

In [13]:
pd.Series({
    "optimal quantile q*": round(q_star, 3),
    "safety margin (MWh)": round(adj, 1),
    "imbalance cost, buy forecast (kEUR)": round(book["cost_fc"].sum() / 1e3),
    "imbalance cost, q* rule (kEUR)": round(short_hours["cost_q"].sum() / 1e3),
    "best % margin": best_margin,
    "saving vs forecast (%)": round(saving, 1),
    "MC 95% CI width (kEUR)": round((np.percentile(sims, 97.5) - np.percentile(sims, 2.5)) / 1e3),
})

optimal quantile q*                       0.378
safety margin (MWh)                    -222.100
imbalance cost, buy forecast (kEUR)    1249.000
imbalance cost, q* rule (kEUR)          583.000
best % margin                            -0.010
saving vs forecast (%)                    7.200
MC 95% CI width (kEUR)                    0.000
dtype: float64

Being long is the expensive side, so we should systematically buy a little *less* than the
forecast. The quantile rule and the percentage-margin grid agree on the direction, the saving is
material, and the Monte Carlo shows the result is very stable. Quantile regression does not help
(worse RMSE than the mean model). Recommendation: bid the forecast minus the optimal margin.